# 00 — Patent Explorer & Selector

Explore the full PatSeer dataset, then browse and flag the patents you want to process.

**Independent of pipeline phases** — run at any time, before or after extraction.

1. **Cell 1** — dataset statistics: see what's in the Excel before filtering
2. **Cell 2** — set your filters and launch the interactive browser
3. **Cell 3 / 4** — check selection count and export to Excel

Every **Select / Deselect** click saves immediately to `logs/selected_patents.json`.

**To feed your selection into the pipeline:** set `subset.mode: "selected"` in `config.yaml` before running `01_extraction.ipynb`.

In [ ]:
import sys; sys.path.insert(0, "..")
import pandas as pd
from src.config_loader import load_config
from src.patents import load_patents

cfg = load_config()
df, missing_df = load_patents(cfg)

print(f"Total patents with valid PDF URL : {len(df)}")
print(f"Patents with no PDF URL          : {len(missing_df)}\n")

print("Record Types:")
print(df["Record Type"].value_counts().to_string())
print("\nLegal Status:")
print(df["Family Legal Status(Dead/Alive)"].value_counts().to_string())
print("\nTech Sub Domain (top 10):")
print(df["Tech Sub Domain"].value_counts().head(10).to_string())
print("\nCPC first code (codes with ≥ 10 patents):")
first_cpc = df["CPC"].fillna("").str.split(r"\s*\|\s*", n=1).str[0].str.strip().str.replace(r"\s+", "", regex=True)
cpc_counts = first_cpc.value_counts()
print(cpc_counts[cpc_counts >= 10].to_string())

In [10]:
from src.selector import PatentSelectorUI

# ── What to show (edit here — no need to touch config.yaml) ──────────────
filters = {
    "cpc_first":    ["B64C29/0041", "B64C29/0083", "B64C29/016"],  # None = show all
    "legal_status": None,   # "ALIVE" | "DEAD" | None
    "record_type":  None,   # "Patent" | None
}
# Set filters=None to browse the entire dataset without any filtering.

ui = PatentSelectorUI(cfg, filters=filters)
ui.show()

Loading patents from Excel…
Loading Excel: /home/vasco/Vasco Workspace/Tese_Vasco_Lnx/Drive_files_to_syncronize/2 - Patente & Validation/3 -Raw_Patent_Exports_PatSeer_&Gold_Standard/1627__dataset_26_05_26.xlsx


/home/vasco/anaconda3/envs/doclayout_yolo/lib/python3.10/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
/home/vasco/anaconda3/envs/doclayout_yolo/lib/python3.10/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


  11 patents have no PDF URL (will appear in status report)
  Loaded 1616 patents with valid PDF URLs.
  Filters applied: 0 patents match.
No patents loaded from Excel.


In [11]:
# How many patents are currently selected?
ui.summary()

Selected: 0 / 0 patents


{'selected': 0, 'total': 0}

In [12]:
# Persist JSON + write logs/selected_patents.xlsx for inspection
ui.export()

KeyError: "None of [Index(['Record Number', 'Title', 'CPC', 'Record Type',\n       'Family Legal Status(Dead/Alive)', 'Tech Sub Domain'],\n      dtype='object')] are in the [columns]"